### Initialisaiton

In [ ]:
# If running in Google Colab; mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
#load requirements
import os
import pathlib
import sys

PROJECT = pathlib.Path("/content/drive/MyDrive/Colab_Notebooks/APJ_project_working_directory") #change as required.
!pip install -r {PROJECT}/requirements.txt

In [ ]:
#reload project dir after kernel restart for requirement dependencies
import pathlib
import os
import sys

DIRECTORY = "/content/drive/MyDrive/Colab_Notebooks/APJ_project_working_directory"
PROJECT = pathlib.Path(DIRECTORY)
os.chdir(PROJECT)
if PROJECT not in sys.path:
  sys.path.append(PROJECT)

#Imports and definitions
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from src.plot_utils import *
from src.data_utils import *

author_subtype_col = "Subclass" 
supertype_col = "Class"
donor_col = 'Donor ID'
subtype_custom = 'APJ_Cats'
genes_of_interest=['APLN','APLNR']
subtype_authors = 'Subclass'
AD_status='Overall AD neuropathological Change'
FIG_DIR='figures'
save_status=True #set to false to view figures
set_paper_theme()

### Pseudobulk comparisons

In [ ]:
#loading sliced SEA-AD data containing only 5 genes.
print("loading data...")
adata_HIP=sc.read_h5ad(f"data/SEA-AD_HIP_Processed.h5ad")
adata_MTG=sc.read_h5ad(f"data/SEA-AD_MTG_Processed.h5ad")
adata_DFC=sc.read_h5ad(f"data/SEA-AD_DFC_Processed.h5ad")
brain_regions=['HIP','MTG','DFC']
adatas={
    "HIP":adata_HIP,
    "MTG":adata_MTG,
    "DFC":adata_DFC
}


In [ ]:
# Validation heatmap for custom categories
markers = ["RBFOX3", "GFAP", "AIF1"]
cats = ["Neuronal", "Astrocyte", "Microglia", "Other"]

fig, axes = plt.subplots(1, len(adatas), figsize=(15, 5), sharey=True)

for i, (ax, (region, adata_r)) in enumerate(zip(axes, adatas.items())):
  if subtype_custom not in adata_r.obs.columns:
    assign_APJ_Cats(adata_r)

  # Standardize category order
  adata_r.obs[subtype_custom] = pd.Categorical(
      adata_r.obs[subtype_custom],
      categories=[c for c in cats if c in adata_r.obs[subtype_custom].values],
  )

  # Dotplot validation
  dp = sc.pl.dotplot(
      adata_r,
      var_names=markers,
      groupby=subtype_custom,
      standard_scale="var",
      cmap="Reds",
      show=False,
      ax=ax,
  )

  # Remove legends from all subplots except the last (far right) panel
  if i != len(adatas) - 1:
    dp["color_legend_ax"].remove()
    dp["size_legend_ax"].remove()
    pos = dp["mainplot_ax"].get_position()

  ax.set_title(f"{region} Region", pad=10)
  ax.grid(True, linestyle="--", alpha=0.3)

plt.suptitle(f"Lineage Marker Validation across {subtype_custom}", y=1.03)
plt.tight_layout()

finalize_panel(
    "Supplementary_Figure_1_Marker_Validation",
    output_dir=FIG_DIR,
    save=save_status,
)

In [ ]:
#UMAPs
for region in brain_regions:
    ax = sc.pl.umap(
        adatas[region],
        color=author_subtype_col,
        title=f"{region} {author_subtype_col} UMAP",
        legend_loc="on data",       # or "right margin"
        legend_fontsize=8,
        frameon=False,
        show=False                  
    )
    for collection in ax.collections:
        collection.set_rasterized(True)
    finalize_panel(filename=f"{region}_{author_subtype_col}_UMAP", save=save_status, output_dir=FIG_DIR)

In [ ]:
import matplotlib.colors as mcolors
#Stacked bar plots
#Stacked bar chart: APJ_Cats proportions per region
meta_all = pd.concat([adata.obs.assign(Region=reg) for reg, adata in adatas.items()])

prop_apj = (
    pd.crosstab(meta_all["Region"], meta_all[subtype_custom], normalize="index")
    .loc[brain_regions] * 100
)


ax = prop_apj.plot(kind="bar", stacked=True, figsize=(5, 5), colormap="Set1")
ax.set(ylabel="Proportion (%)", xlabel="Brain Region", title=f"Cell type Proportions by Region")
ax.legend(title="Cell Type", bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False)
plt.xticks(rotation=0)
finalize_panel(f"proportions_{subtype_custom}_by_region", save=save_status, output_dir=FIG_DIR)


#Stacked bar chart: Neuronal Subclass proportions across regions
neuronal_meta = meta_all[meta_all[subtype_custom] == "Neuronal"]

prop_neuro = (
    pd.crosstab(neuronal_meta["Region"], neuronal_meta[author_subtype_col], normalize="index")
    .loc[brain_regions] * 100
)

palette_40 = list(plt.cm.tab20.colors) + list(plt.cm.tab20b.colors) #increase no. of different colours (there are 21 neuronal categories.)

ax = prop_neuro.plot(kind="bar", stacked=True, figsize=(6, 5), color=palette_40[: len(prop_neuro.columns)])
ax.set(ylabel="Proportion (%)", xlabel="Brain Region", title="Neuronal Subclass Proportions by Region")
ax.legend(title="Subclass", bbox_to_anchor=(1.05, 1), loc="upper left",ncols=2, frameon=False)
plt.xticks(rotation=0)
finalize_panel(f"proportions_neuronal_{author_subtype_col}_by_region", save=save_status, output_dir=FIG_DIR)


### APLNR APLN expression across conditions

In [ ]:
#Loading pseudobulk dataframes stratified by subtype, (and not stratified by subtype general for sex comparison)
regions = ["HIP", "MTG", "DFC"]
AD_status = "Overall AD neuropathological Change"
subtypes_order = ["Microglia", "Astrocyte", "Neuronal"]

#load into a dictionary
print("Loading data...")
def get_pbulk_dfs(regions, selection):
    region_dfs = {}

    for r in regions:
        # Check if saved with or without .csv extension
        filepath = (
            f"data/pseudobulk_{r}_{selection}.csv"
            if os.path.exists(f"data/pseudobulk_{r}_{selection}.csv")
            else f"data/pseudobulk_{r}_{selection}"
        )
        region_dfs[r] = pd.read_csv(filepath)
    return region_dfs

region_dfs_subtype= get_pbulk_dfs(regions=regions, selection="subtype")
region_dfs_general= get_pbulk_dfs(regions=regions, selection="general")

print(f"Loaded all")

In [ ]:
#APLN X NON-CONTINUOUS CATEGORY COMPARISONS 
#Comparison by sex
print("\nGenerating dual-axis subtype plots per region...")
for region, df_subtype in region_dfs_general.items():
    plot_pseudobulk_expression_category(
        df=df_subtype,
        genes_of_interest=genes_of_interest,
        strat_category="Sex",
        FIG_DIR=FIG_DIR,
        title=(f"{region}: expression by Sex"),
        region=region,
        save_status=save_status,
    )

#Comparison by subtype
print("\nGenerating dual-axis subtype plots per region...")
for region, df_subtype in region_dfs_subtype.items():
    plot_pseudobulk_expression_category(
        df=df_subtype,
        genes_of_interest=genes_of_interest,
        strat_category=subtype_custom,
        FIG_DIR=FIG_DIR,
        title=f"{region}: expression by Subtype",
        region=region,
        order=subtypes_order,
        save_status=save_status,
    )

In [ ]:
#cross regional OVERALL comparison by AD stage.
print("\nGenerating cross-region comparison plots...")
print("Overall comparisons")
for gene in genes_of_interest:  # 'APLN', 'APLNR'
    fig, ax = plot_gene_across_regions_by_ad(
        dfs_by_region=region_dfs_general,
        gene=gene,
        cell_type=None,
        subtype_col=subtype_custom,
        ad_col=AD_status,
    )
    finalize_panel(filename=f"overall_{gene}_cross_region_AD.svg", save=save_status, output_dir=FIG_DIR)


#cross regional SUBTYPE comparison by AD stage.
print("subtype-specific comparisons")
for gene in genes_of_interest:  # 'APLN', 'APLNR'
    for c_type in subtypes_order:  # 'Microglia', 'Astrocyte', 'Neuronal'
        fig, ax = plot_gene_across_regions_by_ad(
            dfs_by_region=region_dfs_subtype,
            gene=gene,
            cell_type=c_type,
            subtype_col=subtype_custom,
            ad_col=AD_status,
        )
        finalize_panel(filename=f"{c_type}_{gene}_cross_region_AD.svg", save=save_status, output_dir=FIG_DIR)


### Statistical testing of pseudobulk comparisons

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

#Dependency: run all previous cells first.

# Shared Configurations
genes = ["APLN", "APLNR"]
brain_regions = ["HIP", "MTG", "DFC"]
donor_col = "Donor ID"
subtype_col = subtype_custom  # e.g., 'APJ_Cats'
stage_ranks = {"Not AD": 0, "Low": 1, "Intermediate": 2, "High": 3}
ad_col = "Overall AD neuropathological Change"

# 1. DONOR-LEVEL PROPORTIONS (APJ_Cats & Neuronal Subclasses across Regions)
# CELL ABUNDANCES ARE SIGNIFICANTLY DIFFERENT ACROSS REGIONS, WITH HIP DOMINATED BY A DIFFERENT GROUP OF NEURONS THAN MTG AND DFC


def test_donor_proportions(meta_df, cat_col):
    counts = (
        meta_df.groupby(["Region", donor_col, cat_col], observed=True)
        .size()
        .unstack(fill_value=0)
    )
    props = counts.div(counts.sum(axis=1), axis=0).mul(100).reset_index()

    records = []
    for cat in counts.columns:
        groups = [
            props[props["Region"] == r][cat].dropna().values
            for r in brain_regions
        ]
        groups = [g for g in groups if len(g) > 0]

        # Check: need >= 2 groups AND at least 2 distinct values across all groups
        all_vals = np.concatenate(groups) if len(groups) > 0 else np.array([])

        if len(groups) > 1 and len(np.unique(all_vals)) > 1:
            try:
                stat, p = stats.kruskal(*groups)
            except ValueError:
                stat, p = np.nan, 1.0
        else:
            stat, p = np.nan, 1.0

        res = {
            "Category": cat,
            "Kruskal_Stat": round(stat, 3) if not np.isnan(stat) else np.nan,
            "p_val": p,
        }

        # Add per-region median and mean donor percentages
        for r in brain_regions:
            reg_vals = props[props["Region"] == r][cat]
            res[f"{r}_median_%"] = round(reg_vals.median(), 2)
            res[f"{r}_mean_%"] = round(reg_vals.mean(), 2)

        records.append(res)

    df_out = pd.DataFrame(records)
    if not df_out.empty:
        df_out["FDR"] = multipletests(df_out["p_val"], method="fdr_bh")[1]
    return df_out


df_prop_apj = test_donor_proportions(meta_all, subtype_custom)
df_prop_neuro = test_donor_proportions(neuronal_meta, subtype_col)

# 2. SEX DIFFERENCES PER REGION (Mann-Whitney U)
# SIGNIFICANT: APLN/APLNR EXHIBITS SIGNIFICANT SEXUAL DIMORPHISM IN [REGION] (HIGHER IN MALES/FEMALES)
# NOT SIGNIFICANT: NEITHER APLN NOR APLNR DISPLAYED SIGNIFICANT SEXUAL DIMORPHISM ACROSS EXAMINED REGIONS
records_sex = []
for reg, df in region_dfs_general.items():
    if "Sex" not in df.columns:
        continue
    valid = df[df["Sex"].isin(["Male", "Female", "M", "F"])]

    for g in [g for g in genes if g in valid.columns]:
        m = valid[valid["Sex"].isin(["Male", "M"])][g].dropna()
        f = valid[valid["Sex"].isin(["Female", "F"])][g].dropna()
        stat, p = (
            stats.mannwhitneyu(m, f, alternative="two-sided")
            if len(m) > 1 and len(f) > 1
            else (np.nan, 1.0)
        )
        med_m, med_f = m.median() if len(m) else np.nan, (
            f.median() if len(f) else np.nan
        )

        records_sex.append({
            "Region": reg,
            "Gene": g,
            "N_Male": len(m),
            "N_Female": len(f),
            "Median_Male": round(med_m, 3),
            "Median_Female": round(med_f, 3),
            "log2FC_M_vs_F": round(
                np.log2((med_m + 0.01) / (med_f + 0.01)), 3
            ),
            "MWU_Stat": stat,
            "p_val": p,
        })

df_sex = pd.DataFrame(records_sex)
df_sex["FDR"] = multipletests(df_sex["p_val"], method="fdr_bh")[1]

# 3. SUBTYPE DIFFERENCES PER REGION (Kruskal-Wallis Omnibus)
# SIGNIFICANT: APLN/APLNR DISPLAYS SIGNIFICANT CELL-TYPE SPECIFIC ENRICHMENT ACROSS SUBTYPES IN [REGION]
# NOT SIGNIFICANT: BROADLY EXPRESSED WITHOUT SIGNIFICANT CELL-TYPE SPECIFIC ENRICHMENT
records_subtype = []
subtypes = ["Astrocyte", "Microglia", "Neuronal"]

for reg, df in region_dfs_subtype.items():
    if subtype_col not in df.columns:
        continue

    for g in [g for g in genes if g in df.columns]:
        groups = [
            df[df[subtype_col] == s][g].dropna().values
            for s in subtypes
            if s in df[subtype_col].values
        ]
        all_vals = (
            np.concatenate(groups) if len(groups) > 0 else np.array([])
        )

        # Check: need >= 2 groups and variance across values
        if len(groups) > 1 and len(np.unique(all_vals)) > 1:
            try:
                stat, p = stats.kruskal(*groups)
            except ValueError:
                stat, p = np.nan, 1.0
        else:
            stat, p = np.nan, 1.0

        res = {
            "Region": reg,
            "Gene": g,
            "Kruskal_Stat": round(stat, 3) if not np.isnan(stat) else np.nan,
            "p_val": p,
        }

        # Add median and mean for each cell type
        for s in subtypes:
            vals = (
                df[df[subtype_col] == s][g].dropna()
                if s in df[subtype_col].values
                else []
            )
            res[f"{s}_median"] = (
                round(vals.median(), 3) if len(vals) else np.nan
            )
            res[f"{s}_mean"] = round(vals.mean(), 3) if len(vals) else np.nan

        records_subtype.append(res)

df_subtype = pd.DataFrame(records_subtype)
df_subtype["FDR"] = multipletests(df_subtype["p_val"], method="fdr_bh")[1]


# 4. AD NEUROPATHOLOGY TRAJECTORY (Omnibus, Spearman Trend, High vs Control)
# SIGNIFICANT TREND: MONOTONIC UP/DOWN-REGULATION ALONG PROGRESSIVE AD STAGES
# SIGNIFICANT HIGH VS CTRL ONLY: DISCRETE END-STAGE DYSREGULATION RATHER THAN A GRADUAL TRAJECTORY
def compute_ad_stats(df_dict, restrict_hip_only=False):
    records = []
    items = (
        {"HIP": df_dict.get("HIP", df_dict.get("hip"))}.items()
        if restrict_hip_only
        else df_dict.items()
    )

    for reg, df in items:
        if df is None:
            continue
        cell_types = (
            ["Microglia", "Astrocyte", "Neuronal"]
            if "APJ_Cats" in df.columns
            else [None]
        )

        for ct in cell_types:
            sub = (
                (df[df["APJ_Cats"] == ct] if ct else df)
                .dropna(subset=[ad_col])
                .copy()
            )

            for g in [g for g in genes if g in sub.columns]:
                # 1. Kruskal-Wallis across all 4 stages
                groups = [
                    v[g].values for _, v in sub.groupby(ad_col, observed=True)
                ]
                kw_stat, kw_p = (
                    stats.kruskal(*groups)
                    if len(groups) > 1 and all(len(x) for x in groups)
                    else (np.nan, 1.0)
                )

                # 2. Spearman trend across ordered stages (0 to 3)
                ranked = sub[ad_col].map(stage_ranks)
                valid = ~(ranked.isna() | sub[g].isna())
                rho, p_trend = (
                    stats.spearmanr(ranked[valid], sub.loc[valid, g])
                    if valid.sum() > 2
                    else (np.nan, 1.0)
                )

                records.append({
                    "Region": reg,
                    "Cell_Type": ct or "Overall",
                    "Gene": g,
                    "Trend_Spearman_rho": round(rho, 3),
                    "Trend_p": p_trend,
                    "Kruskal_p": kw_p,
                })

    df_out = pd.DataFrame(records)
    if not df_out.empty:
        df_out["Trend_FDR"] = multipletests(
            df_out["Trend_p"], method="fdr_bh"
        )[1]
    return df_out


df_ad_overall = compute_ad_stats(region_dfs_general, restrict_hip_only=False)
df_ad_hip = compute_ad_stats(region_dfs_subtype, restrict_hip_only=True)

# 5. EXPORT CONSOLIDATED SUPPLEMENTARY WORKBOOK
excel_out = "data/Supplementary_Table_3:_Statistical_testing.xlsx"
with pd.ExcelWriter(excel_out, engine="openpyxl") as writer:
    df_prop_apj.to_excel(writer, sheet_name="Cell_Proportions", index=False)
    df_prop_neuro.to_excel(
        writer, sheet_name="Neuronal_Proportions", index=False
    )
    df_sex.to_excel(writer, sheet_name="Sex_Differences", index=False)
    df_subtype.to_excel(writer, sheet_name="Subtype_Differences", index=False)
    df_ad_overall.to_excel(writer, sheet_name="AD_Staging_Overall", index=False)
    df_ad_hip.to_excel(writer, sheet_name="AD_Staging_HIP", index=False)

print(f"Consolidated analysis saved to: {excel_out}")

### HIP Neuronal and astrocyte GSEA

In [ ]:
import os
import pandas as pd

subtypes_of_interest = ["Astrocyte", "Neuronal"]
genes_of_interest = ["APLN", "APLNR"]

dfs = [
    pd.read_csv(f"data/{gene}_{subtype}_GSEA.csv").assign(
        Subtype=f"{subtype}_{gene}"
    )
    for subtype in subtypes_of_interest
    for gene in genes_of_interest
    if os.path.exists(f"data/{gene}_{subtype}_GSEA.csv")
]

combined_GSEA_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# Pass the full gsea_df, and list both genes in the SUBTYPES array
plot_GSEA_dotplot(
    GENE='Hippocampal APLN & APLNR enriched pathways',            # This just sets the plot title
    SUBTYPES=['Neuronal_APLN', 'Neuronal_APLNR','Astrocyte_APLN','Astrocyte_APLNR'],     # These become your two X-axis columns!
    df=combined_GSEA_df,                     # The combined dataframe we made earlier
    save=save_status
)